# 2. Just-in-time Compilation
so JAX reduces each function into a sequence of **primitive** operations (we can see this using `jax.make_jaxpr()`)

In [1]:
import jax
import jax.numpy as jnp

global_list = []

def log2(x):
    global_list.append(x)
    ln_x = jnp.log(x)
    ln_2 = jnp.log(2.0)
    return ln_x / ln_2

print(jax.make_jaxpr(log2)(3.0))

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


{ lambda ; a:f32[]. let
    b:f32[] = log a
    c:f32[] = log 2.0:f32[]
    d:f32[] = div b c
  in (d,) }


notice that above jaxpr doesn't capture the side-effect (global list append) - this is intentional because JAX transformations are supposed to understand side-effect-free (pure!!!) code. - impure functions are dangerous because their behaviour can be weird...

this is mainly because of **tracing**. when tracing, JAX wraps each argument in a *tracer* object and these objects record all JAX operations performed on them during the function call, then the tracer records are used by JAX to reconstruct the entire function. (the jaxpr is the output of the reconstruction!) **BUT** since tracers do not record the side-effects, they don't appear in the jaxpr, **but still happen during the trace!!**

i.e. `print` is impure

also one thing to note is a jaxpr captures the function *as executed* on the params given to it - so if we have a conditional, the jaxpr will only know about the branch we take:

In [2]:
def square_if_rank_2(x):
    if x.ndim == 2:
        return x ** 2
    else:
        return x
    
print(jax.make_jaxpr(square_if_rank_2)(jnp.array([[1, 2], [1, 2]])))
print(jax.make_jaxpr(square_if_rank_2)(jnp.array([1, 2])))

{ lambda ; a:i32[2,2]. let b:i32[2,2] = integer_pow[y=2] a in (b,) }
{ lambda ; a:i32[2]. let  in (a,) }


## 1.2 JIT Compiling a Function

In [16]:
def selu(x, alpha=1.67, lambda_ = 1.05):
    return x * jnp.where(x > 0, lambda_ * x, lambda_ * alpha * (jnp.exp(x) - 1))

x = jnp.arange(100)

selu(x).at[:10].get()

# problem here is, it's sending one operation at a time to the accelerator... this limits the ability of the XLA compiler to optimize the function :(

Array([ 0.      ,  1.05    ,  4.2     ,  9.45    , 16.8     , 26.25    ,
       37.8     , 51.449997, 67.2     , 85.049995], dtype=float32)

In [19]:
selu_jit = jax.jit(selu)

# precompiling
selu_jit(x).block_until_ready()

%timeit selu_jit(x).block_until_ready()

131 µs ± 7.25 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


wat happened above?

1. We defined selu_jit as the compiled version of selu.

2. We called selu_jit once on x. This is where JAX does its tracing – it needs to have some inputs to wrap in tracers, after all. The jaxpr is then compiled using XLA into very efficient code optimized for your GPU or TPU. Finally, the compiled code is executed to satisfy the call. Subsequent calls to selu_jit will use the compiled code directly, skipping the python implementation entirely. (If we didn’t include the warm-up call separately, everything would still work, but then the compilation time would be included in the benchmark. It would still be faster, because we run many loops in the benchmark, but it wouldn’t be a fair comparison.)

3. We timed the execution speed of the compiled version. (Note the use of block_until_ready(), which is required due to JAX’s Asynchronous dispatch).



## 1.3 Why can't we just JIT everything?

unfortunately, we can't just go around applying `jax.jit()` to every function :(

In [21]:
def f(x):
    if x > 0:
        return x
    else:
        return 2 * x
    
jax.jit(f)(10)

TracerBoolConversionError: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function f at /tmp/ipykernel_2696/1177941068.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument x.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError

In [22]:
# While loop conditioned on x and n.

def g(x, n):
  i = 0
  while i < n:
    i += 1
  return x + i

jax.jit(g)(10, 20)  # Raises an error

TracerBoolConversionError: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function g at /tmp/ipykernel_2696/722961019.py:3 for jit. This concrete value was not available in Python because it depends on the value of the argument n.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError

why not!!!! >:(

so basically when we do: `jax.jit(f)`

it first **traces** your function to build a computation graph.

and during tracing JAX knows things like:
- the shape (100, )
- the dtype (float32)
but not the **actual values**.

In [23]:
def f(x):
    if x > 0:
        return x
    else:
        return 2 * x

so when JAX is tracing above, it sees: `x = ???` - so it can't answer if `x > 0`, therefore it does't know if it should compile:
- `return x`

or 
- `return 2 * x`

hence the ``error``!

In [24]:
while i < n:
    i += 1

NameError: name 'i' is not defined

in above example, during tracing, JAX doesn't know what n is:

is `n = 20`? `n=100`? `n=1`?

so it still can't build the computation graph.

## 1.4 So what CAN JAX branch on?

JAX can use the information that's known during tracing:
- array shape
- dtype
- explicitly static values 

so no **runtime values**.

### 1.4.1 JAX Control Flows

In [25]:
jax.lax.cond(...)
jax.lax.while_loop(...)
jax.lax.fori_loop(...)

# these will become part of the compiled graph

TypeError: _cond() missing 2 required positional arguments: 'true_fun' and 'false_fun'

### 1.4.2 Only JIT the expensive part
instead of doing something like:

In [26]:
while i < n:
    expensive_work()

NameError: name 'i' is not defined

In [27]:
# we can do
@jax.jit
def expensive_work(...):
    ...

SyntaxError: invalid syntax (1422174663.py, line 3)

### 1.4.3 Mark an argument as static

In [29]:
def g(x, n):
    while i < n:
        ...

# then we can mark an argument as static by doing:
jax.jit(g, static_argnames=["n"])

<PjitFunction of <function g at 0x7db920445f80>>

so now we're saying:

"`n` is known when compiling" (so JAX can unroll or compile specifically for n = 20)

In [30]:
# but the catch is...

g(10, 20)   # compile version for n=20

g(10, 30)   # compile AGAIN for n=30

g(10, 40)   # compile AGAIN

# each new value of n requires a new compiled program (so only use this if n only takes on a small # of values)

NameError: name 'i' is not defined

**Rule of thumb**

✅ JIT works great for pure numerical computations where the values don't determine the program structure.

❌ JIT struggles with Python if and while that depend on input values.

✅ Use jax.lax control flow for value-dependent branching inside JIT.

✅ Mark arguments as static only when they take a small, fixed set of values and it's acceptable to recompile for each distinct value.